## ML task definition

**Prediction unit:** one signalized intersection × one prediction day.

**Target:**
- `target = 1` if a KSI collision is registered at the intersection on the next day;
- `target = 0` otherwise.

**Prediction horizon:** 1 day.

For the Week 3 prototype, an intersection is represented by a canonical pair
of `stname1` and `stname2`.

The street names are sorted before creating `intersection_key`, so the order
of `stname1` and `stname2` does not create different identifiers for the same
intersection.

GIS, OSM and ORN are not used at this stage.

KSI events with missing `stname2` are excluded from the Week 3 prototype
because the intersection cannot be identified reliably from the available
street-name fields.

In [46]:
import pandas as pd

ksi_events = pd.read_csv("../data/interim/ksi_events.csv")

ksi_events.head()

,collision_id,accdate,stname1,stname2,stname3,acclass,accloc,traffictl,impactype,visible,light,rdsfcond,road_class,longitude,latitude,division,neighbourhood
0,2006:893184,2006-01-01 02:36:00,WOODBINE AVE,O CONNOR DR,NaN,Non-Fatal Injury,Intersection-Related,No Control,Approaching Head On,Clear,Dark,Wet,Major Arterial,-79.318798,43.699594,55 Division,Woodbine-Lumsden
1,2006:909646,2006-01-01 03:15:00,DANFORTH AVE,WEST LYNN AVE,NaN,Non-Fatal Injury,Intersection-Related,No Control,SMV Other,Snow,Dark with Artificial Lighting,Slush,Major Arterial,-79.316189,43.684873,55 Division,Woodbine Corridor
2,2026:6000734335,2026-04-10 06:10:00,ALBION RD,FINCH AV W,NaN,Non-Fatal Injury,Intersection-Related,Traffic Signal,SMV Other,Rain,Dark,Wet,NaN,-79.589200,43.741099,23 Division,Mount Olive-Silverstone-Jamestown
3,2026:6001470477,2026-07-15 18:33:00,28 CARNFORTH RD,LAWRENCE AV E,5 m East of,Non-Fatal Injury,Non-Intersection,Unknown,Approaching Head On,Clear,Dusk,Dry,NaN,-79.315404,43.739802,33 Division,Victoria Village
4,2026:6001482161,2026-07-17 09:39:00,HIGHWAY 2A E,MEADOWVALE RD,80 m East of,Non-Fatal Injury,Non-Intersection,No Control,Rear End,Clear,Daylight,Dry,NaN,-79.169880,43.780539,43 Division,Centennial Scarborough


In [47]:
strict_accloc = [
    "At Intersection",
    "Four-way",
    "T Intersection",
    "Other Intersection",
    "Roundabout",
]

strict_mask = (
    (ksi_events["traffictl"] == "Traffic Signal")
    &
    (ksi_events["accloc"].isin(strict_accloc))
)

strict_mask.sum()


np.int64(2252)

In [48]:
strict_events = ksi_events[strict_mask]

strict_events[["collision_id", "accdate", "stname1", "stname2", "stname3"]].head()

,collision_id,accdate,stname1,stname2,stname3
6,2006:887568,2006-01-08 15:07:00,PARLIAMENT ST,FRONT ST E,NaN
21,2006:889773,2006-02-02 14:21:00,SPADINA AVE,NASSAU ST,NaN
30,2006:891178,2006-02-14 05:20:00,EGLINTON AVE W,BLACK CREEK DR,NaN
33,2006:893520,2006-02-23 07:00:00,DUNDAS ST W,427 C S DUNDAS RAMP S,NaN
40,2026:6001804531,2026-08-28 18:14:00,LAKE SHORE BV W,JAMESON AV,NaN


In [49]:
strict_events.shape

(2252, 17)

In [50]:
strict_events.stname1.notnull().sum()

np.int64(2252)

In [51]:
strict_events.stname2.notnull().sum()

np.int64(2233)

In [52]:
strict_events.loc[
    strict_events["stname2"].isna(),
    ["collision_id", "stname1", "stname2", "stname3", "accloc", "longitude", "latitude"]
]

,collision_id,stname1,stname2,stname3,accloc,longitude,latitude
301,2007:956470,4400 DUFFERIN Stre,NaN,NaN,At Intersection,-79.466191,43.765444
946,2010:1163934,1181 FINCH AVE W,NaN,NaN,At Intersection,-79.482891,43.765244
1191,2011:1261735,DUFFERIN Stre,NaN,NaN,At Intersection,-79.440991,43.672944
1255,2012:1280520,250 THE EAST MALL,NaN,NaN,At Intersection,-79.552291,43.631444
1719,2014:4003158007,4709 JANE ST,NaN,NaN,At Intersection,-79.520863,43.770902
1855,2015:5001448924,1630 LAWRENCE AVE W,NaN,30 m East of,At Intersection,-79.490841,43.705595
1923,2016:6000171366,3434 LAWRENCE AVE E,NaN,NaN,At Intersection,-79.230435,43.758658
2480,2019:9000941507,350 QUEENS QUAY W,NaN,20 m East of,At Intersection,-79.388638,43.638205
4038,2012:1318227,2700 KINGSTON RD,NaN,NaN,At Intersection,-79.243291,43.715644
4485,2014:142655086,5800 YONGE ST,NaN,NaN,At Intersection,-79.416684,43.784380


In [53]:
prototype_events = strict_events[
    strict_events["stname2"].notna()
]

prototype_events.shape

(2233, 17)

In [54]:
prototype_events = prototype_events.copy()

prototype_events["intersection_key"] = prototype_events.apply(
    lambda row: " | ".join(sorted([row["stname1"], row["stname2"]])),
    axis=1
)

prototype_events[
    ["stname1", "stname2", "intersection_key"]
].head(10)

,stname1,stname2,intersection_key
6,PARLIAMENT ST,FRONT ST E,FRONT ST E | PARLIAMENT ST
21,SPADINA AVE,NASSAU ST,NASSAU ST | SPADINA AVE
30,EGLINTON AVE W,BLACK CREEK DR,BLACK CREEK DR | EGLINTON AVE W
33,DUNDAS ST W,427 C S DUNDAS RAMP S,427 C S DUNDAS RAMP S | DUNDAS ST W
40,LAKE SHORE BV W,JAMESON AV,JAMESON AV | LAKE SHORE BV W
44,MORNINGSIDE AVE,SHEPPARD AVE E,MORNINGSIDE AVE | SHEPPARD AVE E
46,JANE ST,STONG CT,JANE ST | STONG CT
49,BIRCHMOUNT RD,SANWOOD BLVD,BIRCHMOUNT RD | SANWOOD BLVD
55,STEELES AVE E,WARDEN AVE,STEELES AVE E | WARDEN AVE
60,JANE ST,EGLINTON AVE W,EGLINTON AVE W | JANE ST


In [55]:
prototype_events.intersection_key.nunique()

1430

In [56]:
prototype_events["accdate"] = pd.to_datetime(
    prototype_events["accdate"]
)

prototype_events["date"] = prototype_events["accdate"].dt.date

prototype_events[
    ["accdate", "date"]
].head()

,accdate,date
6,2006-01-08 15:07:00,2006-01-08
21,2006-02-02 14:21:00,2006-02-02
30,2006-02-14 05:20:00,2006-02-14
33,2006-02-23 07:00:00,2006-02-23
40,2026-08-28 18:14:00,2026-08-28


In [57]:
prototype_events[
    ["intersection_key", "date"]
].drop_duplicates().shape

(2231, 2)

In [58]:
mask = prototype_events.duplicated(
    subset=["intersection_key", "date"],
    keep=False
)

prototype_events[mask][["collision_id", "intersection_key", "accdate", "date"]].head()


,collision_id,intersection_key,accdate,date
4968,2016:6002166207,COXWELL AVE | GERRARD ST E,2016-12-06 22:20:00,2016-12-06
4969,2016:6002166208,COXWELL AVE | GERRARD ST E,2016-12-06 22:20:00,2016-12-06
7529,2011:1270067,COLLEGE ST | LANSDOWNE AVE,2011-10-18 05:50:00,2011-10-18
7530,2011:1270018,COLLEGE ST | LANSDOWNE AVE,2011-10-18 08:10:00,2011-10-18


In [59]:
prototype_events["prediction_date"] = (
    pd.to_datetime(prototype_events["date"]) - pd.Timedelta(days=1)
)

prototype_events[
    ["date", "prediction_date"]
].head()

,date,prediction_date
6,2006-01-08,2006-01-07
21,2006-02-02,2006-02-01
30,2006-02-14,2006-02-13
33,2006-02-23,2006-02-22
40,2026-08-28,2026-08-27


In [60]:
positive_observations = prototype_events[
    ["intersection_key", "prediction_date"]
].drop_duplicates()

positive_observations.shape

(2231, 2)

In [61]:
positive_observations["target"] = 1
positive_observations.head()

,intersection_key,prediction_date,target
6,FRONT ST E | PARLIAMENT ST,2006-01-07,1
21,NASSAU ST | SPADINA AVE,2006-02-01,1
30,BLACK CREEK DR | EGLINTON AVE W,2006-02-13,1
33,427 C S DUNDAS RAMP S | DUNDAS ST W,2006-02-22,1
40,JAMESON AV | LAKE SHORE BV W,2026-08-27,1


In [62]:
positive_observations.prediction_date = pd.to_datetime(positive_observations.prediction_date)
positive_observations["prediction_date"] = positive_observations["prediction_date"].dt.date
print("Временной промежуток в DF event:", positive_observations.prediction_date.min(), "-", positive_observations.prediction_date.max())

Временной промежуток в DF event: 2006-01-03 - 2026-09-03


In [63]:
positive_observations["prediction_date"] = pd.to_datetime(
    positive_observations["prediction_date"]
)

positive_observations["prediction_date"].dt.year.value_counts().sort_index()

prediction_date
2006    143
2007    129
2008    132
2009    139
2010    138
2011    113
2012    168
2013    110
2014    101
2015     93
2016    107
2017    145
2018    150
2019    149
2020    101
2021     75
2022     98
2023     17
2024     33
2025     62
2026     28
Name: count, dtype: int64

In [64]:
known_before_2018 = prototype_events[
    prototype_events["prediction_date"] < pd.Timestamp("2018-01-01")
]

known_before_2018["intersection_key"].nunique()

1103

In [65]:
positives_2018 = positive_observations[
    positive_observations["prediction_date"].dt.year == 2018
]

positives_2018["prediction_date"].dt.month.value_counts().sort_index()

prediction_date
1      9
2     11
3      5
4     11
5     13
6     11
7     11
8     15
9     21
10    20
11    17
12     6
Name: count, dtype: int64

In [66]:
prototype_dates = pd.date_range(
    start="2018-09-01",
    end="2018-09-30",
    freq="D"
)

prototype_dates

DatetimeIndex(['2018-09-01', '2018-09-02', '2018-09-03', '2018-09-04',
               '2018-09-05', '2018-09-06', '2018-09-07', '2018-09-08',
               '2018-09-09', '2018-09-10', '2018-09-11', '2018-09-12',
               '2018-09-13', '2018-09-14', '2018-09-15', '2018-09-16',
               '2018-09-17', '2018-09-18', '2018-09-19', '2018-09-20',
               '2018-09-21', '2018-09-22', '2018-09-23', '2018-09-24',
               '2018-09-25', '2018-09-26', '2018-09-27', '2018-09-28',
               '2018-09-29', '2018-09-30'],
              dtype='datetime64[us]', freq='D')

In [67]:
len(prototype_dates)

30

In [68]:
print("Strict KSI events:", len(strict_events))
print("Prototype KSI events:", len(prototype_events))
print("Unique intersections:", prototype_events["intersection_key"].nunique())
print("Positive intersection-day observations:", len(positive_observations))

Strict KSI events: 2252
Prototype KSI events: 2233
Unique intersections: 1430
Positive intersection-day observations: 2231


## Step 2 — Build the first X/y dataset

In [69]:
candidate_intersections = known_before_2018.intersection_key.unique()

print(len(candidate_intersections))

1103


In [70]:
candidate_intersections = known_before_2018[
    "intersection_key"
].drop_duplicates()

september_positives = positive_observations[
    (positive_observations["prediction_date"] >= pd.Timestamp("2018-09-01"))
    & (positive_observations["prediction_date"] <= pd.Timestamp("2018-09-30"))
]

covered_positives = september_positives[
    "intersection_key"
].isin(candidate_intersections)

print("Candidate intersections:", len(candidate_intersections))
print("September positives:", len(september_positives))
print("Covered positives:", covered_positives.sum())

Candidate intersections: 1103
September positives: 21
Covered positives: 10


In [71]:
september_2018_positives = positive_observations[
    (positive_observations["prediction_date"].dt.year == 2018)
    &
    (positive_observations["prediction_date"].dt.month == 9)
]

print(len(september_2018_positives))

21


In [72]:
september_2018_positives["intersection_key"].isin(
    candidate_intersections
).sum()

np.int64(10)

In [73]:
uncovered_positives = september_2018_positives[
    ~september_2018_positives["intersection_key"].isin(candidate_intersections)
]

uncovered_positives[
    ["intersection_key", "prediction_date"]
]

,intersection_key,prediction_date
2375,AVENUE RD | CHAPLIN CRES,2018-09-02
2386,BISHOP AVE | WILLOWDALE AVE,2018-09-15
2394,FIREBRACE RD | STEELES AVE E,2018-09-30
5355,MCNICOLL AVE | MIDDLEFIELD RD,2018-09-09
5357,LAWRENCE AVE E | THE DONWAY W,2018-09-11
5362,NORTHCLIFFE BLVD | ST CLAIR AVE W,2018-09-14
5366,CASTLEFIELD AVE | DUFFERIN ST,2018-09-19
5372,CHESTNUT ST | DUNDAS ST W,2018-09-25
5373,JEANETTE ST | ST CLAIR AVE E,2018-09-25
5375,BRIMLEY RD | BRIMWOOD BLVD,2018-09-25


For the Week 3 learning prototype, the candidate universe is limited to signalized intersections that appeared in KSI history before 2018. This is not a complete inventory of Toronto signalized intersections. As a result, some 2018 positive events are outside the prototype scope. A complete spatial inventory will be required in a later project stage.

In [74]:
intersections_df = pd.DataFrame({
    "intersection_key": candidate_intersections
})

dates_df = pd.DataFrame({
    "prediction_date": prototype_dates
})

observation_grid = intersections_df.merge(
    dates_df,
    how="cross"
)

observation_grid.shape

(33090, 2)

In [75]:
observation_grid.head()

,intersection_key,prediction_date
0,FRONT ST E | PARLIAMENT ST,2018-09-01
1,FRONT ST E | PARLIAMENT ST,2018-09-02
2,FRONT ST E | PARLIAMENT ST,2018-09-03
3,FRONT ST E | PARLIAMENT ST,2018-09-04
4,FRONT ST E | PARLIAMENT ST,2018-09-05


In [76]:
covered_september_positives = september_2018_positives[
    september_2018_positives["intersection_key"].isin(candidate_intersections)
]

print(len(covered_september_positives))

10


In [77]:
ml_table = observation_grid.merge(
    covered_september_positives[
        ["intersection_key", "prediction_date", "target"]
    ],
    on=["intersection_key", "prediction_date"],
    how="left"
)

ml_table.shape

(33090, 3)

In [78]:
ml_table["target"].value_counts(dropna=False)

target
NaN    33080
1.0       10
Name: count, dtype: int64

In [79]:
ml_table["target"] = ml_table["target"].fillna(0)
ml_table["target"].value_counts()

target
0.0    33080
1.0       10
Name: count, dtype: int64

In [81]:
ml_table["target"].mean() *100

np.float64(0.030220610456331218)

In [82]:
ml_table["day_of_week"] = ml_table["prediction_date"].dt.dayofweek
ml_table[
    ["prediction_date", "day_of_week"]
].head(10)

,prediction_date,day_of_week
0,2018-09-01,5
1,2018-09-02,6
2,2018-09-03,0
3,2018-09-04,1
4,2018-09-05,2
5,2018-09-06,3
6,2018-09-07,4
7,2018-09-08,5
8,2018-09-09,6
9,2018-09-10,0


In [83]:
ml_table["historical_ksi_count"] = ml_table.apply(
    lambda row: (
        (prototype_events["intersection_key"] == row["intersection_key"])
        &
        (prototype_events["accdate"] < row["prediction_date"])
    ).sum(),
    axis=1
)

ml_table[
    ["intersection_key", "prediction_date", "historical_ksi_count"]
].head(35)

,intersection_key,prediction_date,historical_ksi_count
0,FRONT ST E | PARLIAMENT ST,2018-09-01,1
1,FRONT ST E | PARLIAMENT ST,2018-09-02,1
2,FRONT ST E | PARLIAMENT ST,2018-09-03,1
3,FRONT ST E | PARLIAMENT ST,2018-09-04,1
4,FRONT ST E | PARLIAMENT ST,2018-09-05,1
5,FRONT ST E | PARLIAMENT ST,2018-09-06,1
6,FRONT ST E | PARLIAMENT ST,2018-09-07,1
7,FRONT ST E | PARLIAMENT ST,2018-09-08,1
8,FRONT ST E | PARLIAMENT ST,2018-09-09,1
9,FRONT ST E | PARLIAMENT ST,2018-09-10,1


In [84]:
ml_table["historical_ksi_count"].describe()

count    33090.000000
mean         1.422363
std          0.822943
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          7.000000
Name: historical_ksi_count, dtype: float64

In [85]:
ml_table.head()

,intersection_key,prediction_date,target,day_of_week,historical_ksi_count
0,FRONT ST E | PARLIAMENT ST,2018-09-01,0.0,5,1
1,FRONT ST E | PARLIAMENT ST,2018-09-02,0.0,6,1
2,FRONT ST E | PARLIAMENT ST,2018-09-03,0.0,0,1
3,FRONT ST E | PARLIAMENT ST,2018-09-04,0.0,1,1
4,FRONT ST E | PARLIAMENT ST,2018-09-05,0.0,2,1


In [87]:
X = ml_table[
    ["day_of_week", "historical_ksi_count"]
]

y = ml_table["target"]

X.head()

,day_of_week,historical_ksi_count
0,5,1
1,6,1
2,0,1
3,1,1
4,2,1


In [88]:
y.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: target, dtype: float64

In [89]:
train_mask = ml_table["prediction_date"] <= pd.Timestamp("2018-09-20")
valid_mask = ml_table["prediction_date"] > pd.Timestamp("2018-09-20")

X_train = X[train_mask]
X_valid = X[valid_mask]

y_train = y[train_mask]
y_valid = y[valid_mask]

In [90]:
baseline_train_pred = pd.Series(0, index=y_train.index)
baseline_valid_pred = pd.Series(0, index=y_valid.index)

In [92]:
from sklearn.metrics import accuracy_score

baseline_train_accuracy = accuracy_score(
    y_train,
    baseline_train_pred
)

baseline_valid_accuracy = accuracy_score(
    y_valid,
    baseline_valid_pred
)

print("Train accuracy:", baseline_train_accuracy)
print("Validation accuracy:", baseline_valid_accuracy)

Train accuracy: 0.9996826835902085
Validation accuracy: 0.999728014505893


In [93]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
valid_pred = model.predict(X_valid)

train_accuracy = accuracy_score(
    y_train,
    train_pred
)

valid_accuracy = accuracy_score(
    y_valid,
    valid_pred
)

print("Baseline train accuracy:", baseline_train_accuracy)
print("Baseline validation accuracy:", baseline_valid_accuracy)

print("Random Forest train accuracy:", train_accuracy)
print("Random Forest validation accuracy:", valid_accuracy)

Baseline train accuracy: 0.9996826835902085
Baseline validation accuracy: 0.999728014505893
Random Forest train accuracy: 0.9996826835902085
Random Forest validation accuracy: 0.999728014505893


## Week 3 conclusion

The first ML task for the Toronto pilot was formulated as next-day KSI risk
prediction for a `signalized intersection × day` observation.

### Prototype dataset

- Prediction horizon: 1 day.
- Spatial unit: canonical intersection name based on `stname1` and `stname2`.
- GIS, OSM and ORN were not used.
- The learning prototype uses signalized intersections observed in KSI history
  before 2018.
- Prototype period: September 2018.
- Observation grid: 33,090 intersection-day rows.
- Positive observations: 10.
- Target prevalence: approximately 0.03%.

### Features

The first feature set contains:

- `day_of_week`;
- `historical_ksi_count`, calculated using only KSI events strictly before the
  prediction date.

Event-time fields such as `light`, `visible` and `rdsfcond` were not used
because they would not be known for all observations at prediction time and
could introduce leakage.

### Validation

A chronological split was used:

- September 1-20, 2018: training;
- September 21-30, 2018: validation.

Rows were not randomly shuffled.

### Results

Always-zero baseline:

- Train accuracy: 0.9996826835902085
- Validation accuracy: 0.999728014505893

Random Forest:

- Train accuracy: 0.9996826835902085
- Validation accuracy: 0.999728014505893

Random Forest did not improve accuracy over the always-zero baseline.

The experiment demonstrates why accuracy is misleading for this extremely
imbalanced rare-event problem. Later project stages should use metrics designed
for rare-event ranking and probability quality rather than treating accuracy as
the primary metric.

### Limitations

- The candidate intersection set is not a complete inventory of Toronto
  signalized intersections.
- It contains only intersections observed in historical KSI data before 2018.
- Only 10 of the 21 September 2018 positive observations fall inside this
  prototype universe.
- The Week 3 dataset is a learning prototype and is not the final Project A
  modelling dataset.
- The feature set is intentionally minimal.
- No hyperparameter tuning was performed.
- Final validation, negative-observation construction and spatial coverage will
  be redesigned in later project stages.